# Three Decision Models

### d1, Jev and OpenAI's Decisions API, asked the seven swaps' questions

Notebooks 01 to 07 used one decision model, Liquid's `d1-2B`, running on this laptop. Two more answer the same kinds of question as hosted APIs: TypeSafe's **Jev** and OpenAI's **Decisions API** with `gpt-6-luna`. Nearly every decision model answers the same request shape (a state, named questions, allowed answers), so the model under the architecture is a configuration choice. This notebook tests that: the exact questions of the seven swaps, on the same labels, sent to all three.

| | d1 | Jev | OpenAI Decisions |
|---|---|---|---|
| Maker | Liquid AI | TypeSafe | OpenAI |
| Model here | `LiquidAI/d1-2B`, open weights | `jev-1.14.0` | `gpt-6-luna` |
| Runs | on this laptop | hosted | hosted |
| Call | `model.system_one(state, questions)` | `POST api.typesafe.ai/v1/systemone` | `POST api.openai.com/v1/decisions` |
| Question types | noul, choice, score | noul, choice, score | predicate, choice, score |
| Input price per million tokens | free locally, $0.02 as Liquid's hosted d1.1 | $0.042 | $0.10 uncached |

Output tokens are not billed by Liquid or OpenAI, and TypeSafe states only an input price.

| Part | What you will see |
|---|---|
| 1 · The three models | One question to each, the two request shapes, and the time per request |
| 2 · The seven swaps | Every swap's decision, asked of every model |
| 3 · The table | Accuracy, cost and time, side by side |

**Live path.**

| Section | What to show | Minutes |
|---|---|---|
| ⭐ 1.2 | One question, three answers | 2 |
| ⭐ 1.3 | Two request shapes, one answer shape | 2 |
| ⭐ 3.1 | The table | 3 |

### How decision models are built, and who builds them

The three models here are built differently: a frozen backbone with a trained head, a pointer head, an encoder, or a dual encoder. What they share is the request: a state, named questions and the answers allowed. That shared shape is what lets this notebook swap the model with one setting.

<img src="img/reference/p19_how_a_decision_model_is_built.png" alt="How a decision model is built: request, template, backbone, one prefill pass, decision head, probabilities; and five ways the field builds the head." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 19. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p20_ecosystem.png" alt="Seven weeks, ten launches, one request shape: the decision-model releases from Jev to d1." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 20. The full PDF is in <code>docs/</code>.</sub>

### The comparison

<img src="img/09_decision_model_providers.png" alt="The same state and questions go to d1 on this laptop, to Jev over HTTP, and through a translation step to OpenAI's Decisions API. All three answers come back in one shape and are scored the same way." width="600">

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    Q["Same state and questions<br/>noul · choice · score"]:::code --> D["d1-2B<br/>this laptop"]:::dm
    Q --> J["Jev<br/>/v1/systemone"]:::dm
    Q --> T["Translate<br/>predicate · choice · score"]:::code --> O["gpt-6-luna<br/>/v1/decisions"]:::dm
    D --> A["One answer shape<br/>P(yes) · pick · level"]:::code
    J --> A
    O --> A
    A --> M[(Scored on the<br/>seven label sets)]:::db
    classDef user fill:#1e1b4b,stroke:#a78bfa,color:#ede9fe
    classDef app fill:#0c2a4d,stroke:#60a5fa,color:#dbeafe
    classDef llm fill:#3b1d0a,stroke:#fb923c,color:#ffedd5
    classDef dm fill:#052e2b,stroke:#2dd4bf,color:#ccfbf1
    classDef db fill:#3b0a1e,stroke:#f472b6,color:#fce7f3
    classDef mem fill:#1f2a07,stroke:#a3e635,color:#ecfccb
    classDef tool fill:#2e1065,stroke:#c084fc,color:#f3e8ff
    classDef gate fill:#3f2a00,stroke:#fbbf24,color:#fef3c7
    classDef code fill:#111827,stroke:#94a3b8,color:#e5e7eb
    classDef bad fill:#3b0a0a,stroke:#f87171,color:#fee2e2
```

</details>

# Part 1 · The Three Models

## 1.1 · Keys and the local model

`harness.d1` sends the same call to any of the three: `d1.using("jev")` or `d1.using("openai")` switches the model for a block of code, and answers always come back in d1's shape. For d1 on this laptop, `decide` calls Liquid's own `model.system_one`, which downloads with the model; for Jev and OpenAI it sends an HTTP request. Notebook 00, section 1.4, shows the split.

In [ ]:
import sys
sys.path.insert(0, "..")

import harness
from harness import measure

print("keys set:", harness.load_env())
measure.use_notebook("09_decision_model_providers")

keys set: ['OPENAI_API_KEY', 'TYPESAFE_API_KEY', 'HF_TOKEN']


In [ ]:
import time
import numpy as np
import pandas as pd
from harness import d1

MODELS = {"d1": "local", "Jev": "jev", "OpenAI Decisions": "openai"}
d1.model()
print("d1 on", d1.device())

d1 on mps


## 1.2 · ⭐ One question, three answers

One support message and three questions: a yes or no, a pick from three teams, and a level on a three-step scale.

In [ ]:
QUESTIONS = {
    "billing": d1.noul("Is this about a payment problem?"),
    "team": d1.choice("Which team should handle this?", {
        "billing": "Payments, invoices and refunds.",
        "technical": "Problems using the product.",
        "shipping": "Delivery and tracking."}),
    "urgency": d1.score("How urgent is this?", [
        "It can wait a week.", "It should be handled this week.", "It needs a reply today."]),
}
MESSAGE = "I was charged twice for the twelve seats, and the invoice is due on Friday."

first = {}
for name, backend in MODELS.items():
    with d1.using(backend):
        answer = d1.decide(MESSAGE, QUESTIONS)
    first[name] = {"P(billing)": answer["billing"]["noul"], "team": answer["team"]["choice"],
                   "team confidence": answer["team"]["confidence"], "urgency level": answer["urgency"]["score"]}
pd.DataFrame(first).T.round(3)

,P(billing),team,team confidence,urgency level
d1,0.8098,billing,0.928918,1.0
Jev,0.97,billing,1.0,1.21
OpenAI Decisions,0.99,billing,1.0,1.15


**What this shows:** all three pick `billing` and put the urgency between "this week" and "today". The numbers differ: d1 is less sure this is a payment problem (0.81) than Jev and OpenAI (0.97 and 0.99). **Each model's probabilities sit on its own scale**, so a threshold set for one does not carry over to another.

## 1.3 · ⭐ Two request shapes, one answer shape

Jev takes the same request as d1: a `state` and a dictionary of named questions, so only the URL changes. OpenAI takes the same ideas in a different shape: `input` instead of `state`, an ordered list of questions, `predicate` instead of `noul`, and options as objects. The harness translates, and translates the answers back.

In [ ]:
import json
from harness.d1 import _openai_question

print("d1 and Jev:\n" + json.dumps(QUESTIONS["team"], indent=1))
print("\nOpenAI:\n" + json.dumps(_openai_question("team", QUESTIONS["team"]), indent=1))

d1 and Jev:
{
 "type": "choice",
 "instructions": "Which team should handle this?",
 "criteria": {
  "billing": "Payments, invoices and refunds.",
  "technical": "Problems using the product.",
  "shipping": "Delivery and tracking."
 }
}

OpenAI:
{
 "type": "choice",
 "name": "team",
 "choices": [
  {
   "value": "billing",
   "description": "Payments, invoices and refunds."
  },
  {
   "value": "technical",
   "description": "Problems using the product."
  },
  {
   "value": "shipping",
   "description": "Delivery and tracking."
  }
 ],
 "instructions": "Which team should handle this?"
}


## 1.4 · Time per request

Twenty single requests to each model, one at a time, with the routing question from swap 4. The median is the time a user would wait for one decision.

In [ ]:
from harness import questions as Q

routing = harness.labels("memory_routing")["turns"]
latency = {}
for name, backend in MODELS.items():
    with d1.using(backend):
        d1.decide(routing[0]["turn"], Q.ROUTES)
        times = []
        for turn in routing[1:21]:
            started = time.perf_counter()
            d1.decide(turn["turn"], Q.ROUTES)
            times.append((time.perf_counter() - started) * 1000)
    latency[name] = float(np.median(times))
pd.Series(latency, name="median ms per request").round(0).to_frame()

,median ms per request
d1,151.0
Jev,190.0
OpenAI Decisions,238.0


**What this shows:** one decision takes about 150 ms with d1 on this laptop, and about 190 ms and 240 ms over the network with Jev and OpenAI. All three are a fraction of an LLM call: notebook 04's routing prompt took about 620 ms.

# Part 2 · The Seven Swaps

## 2.1 · The items

Each swap's decision, built exactly as its notebook built it, with the question text from `harness/questions.py` (the same words the notebooks use). Each entry is a list of (state, questions) requests and a function that scores the answers with the swap's own measure.

In [ ]:
import inspect
import shutil
from langchain_openai import OpenAIEmbeddings
from memorizz import FileSystemConfig, FileSystemProvider, KnowledgeBase, Toolbox
from memorizz.benchmarks.jev_reranking import question_payload, scores_from_answers
from memorizz.enums import MemoryType
from harness import demo_tools
from harness.spans import boundary_scores, retrieval_scores, sentence_starts, split_sentences, with_offsets

embeddings = OpenAIEmbeddings(model=harness.EMBEDDING_MODEL)

def embed(texts):
    vectors = np.array(embeddings.embed_documents(texts))
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

SWAPS = {}

In [ ]:
# Swap 1 · chunking: one question per gap; boundary F1 weighs real topic changes found against extra cuts.
text, gold = harness.demo()
sentences = split_sentences(text)
starts = sentence_starts(text, sentences)
real_changes = [t["start"] for t in gold["topics"][1:]]

def score_chunking(answers):
    cuts = [starts[i + 1] for i, a in enumerate(answers) if a["boundary"]["noul"] >= 0.5]
    found = boundary_scores(cuts, real_changes, starts, tolerance=1)
    return found["f1"], f"{found['true_positive']} of {len(real_changes)} found, {found['cuts'] - found['true_positive']} extra"

SWAPS["1 · chunking"] = ([(Q.boundary_state(sentences, i), {"boundary": Q.CHUNK_BOUNDARY}) for i in range(len(sentences) - 1)],
                         score_chunking, "boundary F1")

In [ ]:
# Swap 2 · reranking: one question per candidate, 40 questions x 20 candidates; scored by nDCG@5.
def sentence_pairs(corpus, **kwargs):
    s = sentence_starts(corpus)
    edges = s[::2] + [len(corpus)]
    return [corpus[a:b].strip() for a, b in zip(edges, edges[1:])]

ROOT = harness.STORE / "providers"
shutil.rmtree(ROOT, ignore_errors=True)
provider = FileSystemProvider(FileSystemConfig(root_path=ROOT))
KnowledgeBase(memory_provider=provider).ingest_knowledge(text, namespace="pairs", chunking_strategy=sentence_pairs)
spans_of = {row["text"]: (row["start"], row["end"]) for row in with_offsets(text, sentence_pairs(text))}
eval_questions = harness.eval_questions()
pools = [[row["content"] for row in provider.retrieve_by_query(q["question"], memory_store_type=MemoryType.KNOWLEDGE_BASE,
                                                               limit=20, namespace="pairs")] for q in eval_questions]

def relevant(chunk, q):
    return any(retrieval_scores([spans_of[chunk]], [tuple(s)])["recall"] >= 0.5 for s in q["spans"])

def score_reranking(answers):
    scores, at = [], 0
    for q, pool in zip(eval_questions, pools):
        p = [scores_from_answers(a, 1, "noul")[0] for a in answers[at:at + len(pool)]]
        at += len(pool)
        hits = [relevant(pool[i], q) for i in np.argsort(-np.asarray(p), kind="stable")]
        total = max(sum(relevant(c, q) for c in spans_of), 1)
        dcg = sum(h / np.log2(r + 2) for r, h in enumerate(hits[:5]))
        scores.append(dcg / sum(1 / np.log2(r + 2) for r in range(min(total, 5))))
    return float(np.mean(scores)), f"nDCG@5 {np.mean(scores):.3f}"

SWAPS["2 · reranking"] = ([question_payload(q["question"], [c], "noul") for q, pool in zip(eval_questions, pools) for c in pool],
                          score_reranking, "nDCG@5")

In [ ]:
# Swap 3 · semantic cache: the three closest cached answers per query; a hit is the best P >= 0.5.
cache = harness.labels("cache")
by_id = {c["id"]: c for c in cache["cached"]}
cached_vectors = embed([c["question"] for c in cache["cached"]])
query_vectors = embed([q["query"] for q in cache["queries"]])
candidates = [[cache["cached"][j]["id"] for j in np.argsort(-(cached_vectors @ v))[:3]] for v in query_vectors]

def score_cache(answers):
    right, at = [], 0
    for q, cands in zip(cache["queries"], candidates):
        p, best = max((a["serves"]["noul"], c) for a, c in zip(answers[at:at + 3], cands))
        at += 3
        served = best if p >= 0.5 else None
        right.append(served == q["serves"])
    return float(np.mean(right)), f"{sum(right)} of {len(right)} right"

SWAPS["3 · semantic cache"] = ([({"new_question": q["query"], "cached_answer": by_id[c]["answer"]}, {"serves": Q.CACHE_SERVES})
                                for q, cands in zip(cache["queries"], candidates) for c in cands],
                               score_cache, "right decisions")

In [ ]:
# Swap 4 · memory routing: three yes-or-no questions per turn; right when the set of stores matches.
def score_routing(answers):
    right = [sorted(k for k in Q.ROUTES if a[k]["noul"] >= 0.5) == sorted(t["stores"]) for a, t in zip(answers, routing)]
    return float(np.mean(right)), f"{sum(right)} of {len(right)} right"

SWAPS["4 · memory routing"] = ([(t["turn"], Q.ROUTES) for t in routing], score_routing, "right route")

# Swap 5 · consolidation: one Choice over five relations; right when the action matches.
dedup = harness.labels("dedup")
def score_dedup(answers):
    right = [dedup["actions"].get(a["relation"]["choice"]) == dedup["actions"][p["relation"]] for a, p in zip(answers, dedup["pairs"])]
    return float(np.mean(right)), f"{sum(right)} of {len(right)} right"

SWAPS["5 · consolidation"] = ([({"stored_memory": p["stored"], "new_memory": p["new"]}, {"relation": Q.DEDUP}) for p in dedup["pairs"]],
                              score_dedup, "right action")

# Swap 6 · extraction: the gate; right when it lets durable turns through and stops the rest.
extraction = harness.labels("extraction")["turns"]
def score_extraction(answers):
    right = [(a["durable"]["noul"] >= 0.5) == t["durable"] for a, t in zip(answers, extraction)]
    typed = [a["type"]["choice"] == t["type"] for a, t in zip(answers, extraction) if t["durable"]]
    return float(np.mean(right)), f"gate {sum(right)} of {len(right)}, type {sum(typed)} of {len(typed)}"

SWAPS["6 · extraction"] = ([(t["turn"], Q.EXTRACTION) for t in extraction], score_extraction, "gate right")

In [ ]:
# Swap 7 · tool selection: a Choice over the embedding top 5, plus none.
TOOLS_ROOT = harness.STORE / "providers_tools"
shutil.rmtree(TOOLS_ROOT, ignore_errors=True)
tool_store = FileSystemProvider(FileSystemConfig(root_path=TOOLS_ROOT))
demo_tools.bind(tool_store)
toolbox = Toolbox(memory_provider=tool_store)
for tool in demo_tools.TOOLS:
    toolbox.register_tool(tool)
tool_turns = harness.labels("tools")["turns"]
top5 = [[row["name"] for row in toolbox.get_most_similar_tools(t["turn"], limit=5)] for t in tool_turns]

def score_tools(answers):
    right = [a["tool"]["choice"] == t["tool"] for a, t in zip(answers, tool_turns)]
    return float(np.mean(right)), f"{sum(right)} of {len(right)} right"

SWAPS["7 · tool selection"] = ([(t["turn"], {"tool": Q.tool_question(names)}) for t, names in zip(tool_turns, top5)],
                               score_tools, "right tool first")
{name: len(items) for name, (items, _, _) in SWAPS.items()}

{'1 · chunking': 153,
 '2 · reranking': 800,
 '3 · semantic cache': 144,
 '4 · memory routing': 58,
 '5 · consolidation': 50,
 '6 · extraction': 60,
 '7 · tool selection': 75}

## 2.2 · Ask every model

Every model answers every request. Hosted calls run eight at a time and d1 runs in batches on the GPU, so the seconds below are for the whole set, not per request (1.4 has that). Cost is each provider's input price on the tokens it reports, with d1 priced at Liquid's hosted rate. OpenAI's API can refuse a question; a refusal comes back as P = 0.5 for a yes-or-no question and as no pick for a Choice, and the count is in the table.

In [ ]:
results, kept = [], {}
for swap, (items, score, measure_name) in SWAPS.items():
    for name, backend in MODELS.items():
        d1.usage.update(requests=0, questions=0, input_tokens=0, refusals=0)
        started = time.perf_counter()
        with d1.using(backend):
            answers = d1.decide_batch(items)
        seconds = time.perf_counter() - started
        usd = measure.cost(f"{name}:{swap}", d1.usage["input_tokens"], 0, model=d1.price_model(backend), count=len(items))
        kept[swap, name] = answers
        value, detail = score(answers)
        measure.metric(name, measure_name, value, swap=swap)
        results.append({"swap": swap, "model": name, "measure": measure_name, "score": round(value, 3), "detail": detail,
                        "requests": len(items), "seconds": round(seconds, 1), "usd per 1,000 requests": usd / len(items) * 1000,
                        "refusals": d1.usage["refusals"]})
    print(swap, "done", flush=True)

1 · chunking done


2 · reranking done


3 · semantic cache done


4 · memory routing done


5 · consolidation done


6 · extraction done


7 · tool selection done


# Part 3 · The Table

## 3.1 · ⭐ Every swap, every model

In [ ]:
frame = pd.DataFrame(results)
table = frame.pivot(index="swap", columns="model", values="score")[list(MODELS)]
table.insert(0, "measure", frame.groupby("swap")["measure"].first())
table["best"] = [", ".join(m for m in MODELS if row[m] == max(row[x] for x in MODELS)) for _, row in table.iterrows()]
table

model,measure,d1,Jev,OpenAI Decisions,best
swap,,,,,
1 · chunking,boundary F1,0.476,0.318,0.311,d1
2 · reranking,nDCG@5,0.890,0.903,0.942,OpenAI Decisions
3 · semantic cache,right decisions,0.979,0.979,1.000,OpenAI Decisions
4 · memory routing,right route,0.672,0.379,0.431,d1
5 · consolidation,right action,0.880,0.900,0.900,"Jev, OpenAI Decisions"
6 · extraction,gate right,0.983,1.000,1.000,"Jev, OpenAI Decisions"
7 · tool selection,right tool first,0.960,0.960,0.973,OpenAI Decisions


**What this shows:**
- **No model wins everywhere. OpenAI Decisions is best or tied best on five of the seven swaps** (reranking, the cache, consolidation, extraction, tools); **d1 is best on chunking and memory routing.**
- **On five swaps the three land within a few items of each other:** the cache (47 to 48 of 48), consolidation (44 to 45 of 50), the extraction gate (59 to 60 of 60), tools (72 to 73 of 75) and reranking (nDCG@5 0.89 to 0.94).
- **Two swaps separate them.** In chunking, Jev and OpenAI find every topic change but make about 30 extra cuts at P ≥ 0.5, because their probabilities run higher. In memory routing d1 is far ahead (0.67 against 0.38 and 0.43), and the criteria were written and checked with d1 in notebook 04. 3.2 shows what went wrong.

In [ ]:
frame.pivot(index="swap", columns="model", values="detail")[list(MODELS)]

model,d1,Jev,OpenAI Decisions
swap,,,
1 · chunking,"5 of 7 found, 9 extra","7 of 7 found, 30 extra","7 of 7 found, 31 extra"
2 · reranking,nDCG@5 0.890,nDCG@5 0.903,nDCG@5 0.942
3 · semantic cache,47 of 48 right,47 of 48 right,48 of 48 right
4 · memory routing,39 of 58 right,22 of 58 right,25 of 58 right
5 · consolidation,44 of 50 right,45 of 50 right,45 of 50 right
6 · extraction,"gate 59 of 60, type 31 of 36","gate 60 of 60, type 34 of 36","gate 60 of 60, type 36 of 36"
7 · tool selection,72 of 75 right,72 of 75 right,73 of 75 right


## 3.2 · Where they differ most: memory routing

The routing questions ask one yes or no per memory type, and a route is right only when exactly the needed stores say yes. Two numbers show what kind of mistake each model makes: how often every needed store said yes, and how many extra stores said yes per turn.

In [ ]:
def routing_profile(answers):
    routed = [{k for k in Q.ROUTES if a and a[k]["noul"] >= 0.5} for a in answers]
    return {"right route": np.mean([r == set(t["stores"]) for r, t in zip(routed, routing)]),
            "every needed store said yes": np.mean([set(t["stores"]) <= r for r, t in zip(routed, routing)]),
            "extra stores per turn": np.mean([len(r - set(t["stores"])) for r, t in zip(routed, routing)]),
            "stores per turn": np.mean([len(r) for r in routed]),
            "no-memory turns right": f"{sum(not r for r, t in zip(routed, routing) if not t['stores'])} of {sum(not t['stores'] for t in routing)}"}

pd.DataFrame({m: routing_profile(kept["4 · memory routing", m]) for m in MODELS}).T.round(2)

,right route,every needed store said yes,extra stores per turn,stores per turn,no-memory turns right
d1,0.672414,0.87931,0.241379,1.12069,11 of 12
Jev,0.37931,0.948276,0.637931,1.586207,9 of 12
OpenAI Decisions,0.431034,0.793103,0.37931,1.172414,12 of 12


**What this shows:** the hosted models fail in opposite directions. **Jev says yes too often**: 0.64 extra stores per turn, so it searches every needed store on 0.95 of turns but rarely only those. **OpenAI says no more often**: it misses a needed store on about one turn in five (0.79), though it searches nothing on every turn that needs nothing (12 of 12). d1 sits between them. The same question at the same 0.5 line gives three different trade-offs, and a few labelled turns show where each model's line belongs.

## 3.3 · Cost and time for the whole set

In [ ]:
costs = frame.groupby("model").agg(requests=("requests", "sum"), seconds=("seconds", "sum"), refusals=("refusals", "sum"))
costs["usd for all seven sets"] = [sum(r["usd per 1,000 requests"] * r["requests"] / 1000 for r in results if r["model"] == m)
                                    for m in costs.index]
costs["median ms per request (1.4)"] = [round(latency[m]) for m in costs.index]
costs.loc[list(MODELS)].round({"seconds": 0, "usd for all seven sets": 4})

,requests,seconds,refusals,usd for all seven sets,median ms per request (1.4)
model,,,,,
d1,1340,121.0,0,0.0037,151
Jev,1340,33.0,0,0.0222,190
OpenAI Decisions,1340,40.0,0,0.0363,238


**What this shows:** for all 1,340 requests, d1 would cost under half a cent at Liquid's hosted rate (and nothing on this laptop), Jev about two cents and OpenAI about four. Eight at a time, the hosted models finish the whole set in 33 and 40 seconds; d1 takes about two minutes on this laptop's GPU. OpenAI refused nothing.

> 📌 **Key Takeaways**
> - The same request shape works across all three, so the model under the architecture is a configuration choice.
> - On most swaps the three land within a few items of each other. OpenAI Decisions was best or tied on five of seven, d1 on two.
> - Probabilities sit on each model's own scale. Chunking and routing show it: set the threshold per model, on your own labels, before you compare.
> - Cost and where it runs differ more than accuracy: for 1,340 decisions, d1 is free on your own machine, Jev costs about two cents and OpenAI about four.